# Hopfield Network — Implementation from Scratch
# Hopfield 네트워크 — 직접 구현

**Paper / 논문**: J. J. Hopfield, "Neural Networks and Physical Systems with Emergent Collective Computational Abilities" (1982)

이 노트북에서는 Hopfield 네트워크의 핵심 알고리즘을 NumPy로 직접 구현하고, 논문의 주요 결과를 재현합니다.

This notebook implements the Hopfield network algorithm from scratch using NumPy and reproduces the paper's key results.

**Contents / 목차:**
1. **Part 1**: Core Implementation — Hebbian 저장 + 비동기 갱신 / Hebbian storage + asynchronous update
2. **Part 2**: Energy Function — 에너지 단조 감소 시각화 / Energy monotonic decrease visualization
3. **Part 3**: Pattern Storage & Retrieval — 패턴 저장과 복원 데모 / Storage and retrieval demo
4. **Part 4**: Attractor Basins — 끌개 영역 시각화 / Attractor basin visualization
5. **Part 5**: Storage Capacity — 용량 한계 실험 ($\sim 0.15N$) / Capacity limit experiment
6. **Part 6**: Hamming Distance Analysis — 복원 성공률 vs 거리 / Retrieval success vs distance
7. **Part 7**: Spurious States — 가짜 기억 탐색 / Exploring spurious states
8. **Part 8**: Connection to Modern Methods — 현대 기법과의 연결 / Link to modern methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from typing import Optional

np.random.seed(42)

# Plot style
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: Core Implementation / 핵심 구현

Hopfield 네트워크의 두 핵심 요소를 구현합니다:

We implement the two core components of the Hopfield network:

**1. Hebbian 저장 규칙 (Eq. 2) / Hebbian Storage Prescription:**
$$T_{ij} = \sum_s (2V_i^s - 1)(2V_j^s - 1), \quad T_{ii} = 0$$

**2. 비동기 갱신 규칙 (Eq. 1) / Asynchronous Update Rule:**
$$V_i \rightarrow \begin{cases} 1 & \text{if } \sum_{j \neq i} T_{ij} V_j > 0 \\ 0 & \text{if } \sum_{j \neq i} T_{ij} V_j < 0 \end{cases}$$

**3. 에너지 함수 (Eq. 7) / Energy Function:**
$$E = -\frac{1}{2} \sum_{i \neq j} T_{ij} V_i V_j$$

In [ ]:
class HopfieldNetwork:
    """Hopfield network with Hebbian learning (Hopfield, 1982).

    Implements the exact algorithm from the paper: binary neurons {0, 1},
    Hebbian storage prescription (Eq. 2), asynchronous update (Eq. 1),
    and energy function (Eq. 7).
    """

    def __init__(self, n_neurons: int) -> None:
        """Initialize network with n_neurons binary neurons.

        Args:
            n_neurons: Number of neurons N in the network.
        """
        self.n_neurons = n_neurons
        self.weights = np.zeros((n_neurons, n_neurons))
        self.patterns: list[np.ndarray] = []

    def store(self, patterns: np.ndarray) -> None:
        """Store patterns using Hebbian prescription (Eq. 2).

        T_ij = sum_s (2*V_i^s - 1)(2*V_j^s - 1), T_ii = 0

        Args:
            patterns: Array of shape (n_patterns, n_neurons) with values in {0, 1}.
        """
        self.patterns = [p.copy() for p in patterns]
        # Convert {0, 1} -> {-1, +1}
        bipolar = 2 * patterns - 1  # shape: (n_patterns, N)
        # Outer product sum: T_ij = sum_s bipolar_i^s * bipolar_j^s
        self.weights = bipolar.T @ bipolar  # shape: (N, N)
        # Zero diagonal: T_ii = 0
        np.fill_diagonal(self.weights, 0)

    def energy(self, state: np.ndarray) -> float:
        """Compute energy E = -0.5 * sum_{i!=j} T_ij V_i V_j (Eq. 7).

        Args:
            state: Binary state vector of shape (n_neurons,).

        Returns:
            Energy scalar.
        """
        return -0.5 * state @ self.weights @ state

    def update_neuron(self, state: np.ndarray, i: int) -> np.ndarray:
        """Asynchronously update neuron i according to Eq. 1.

        Args:
            state: Current binary state vector.
            i: Index of neuron to update.

        Returns:
            Updated state vector (modified in place).
        """
        h_i = self.weights[i] @ state
        if h_i > 0:
            state[i] = 1
        elif h_i < 0:
            state[i] = 0
        # If h_i == 0, keep current state (tie-breaking)
        return state

    def retrieve(
        self,
        probe: np.ndarray,
        max_steps: int = 1000,
        track_energy: bool = False,
    ) -> dict:
        """Retrieve a stored pattern from a (possibly corrupted) probe.

        Uses asynchronous random update until convergence or max_steps.

        Args:
            probe: Initial state vector (corrupted pattern).
            max_steps: Maximum number of neuron updates.
            track_energy: If True, record energy at each step.

        Returns:
            Dict with keys: "state" (final state), "steps" (number of updates),
            "converged" (bool), "energies" (list, if track_energy).
        """
        state = probe.copy()
        energies = [self.energy(state)] if track_energy else []

        for step in range(max_steps):
            i = np.random.randint(self.n_neurons)
            old_val = state[i]
            self.update_neuron(state, i)

            if track_energy:
                energies.append(self.energy(state))

            # Check convergence: full sweep with no change
            if (step + 1) % self.n_neurons == 0:
                changed = False
                for j in range(self.n_neurons):
                    h_j = self.weights[j] @ state
                    expected = 1 if h_j > 0 else (0 if h_j < 0 else state[j])
                    if state[j] != expected:
                        changed = True
                        break
                if not changed:
                    return {
                        "state": state,
                        "steps": step + 1,
                        "converged": True,
                        "energies": energies,
                    }

        return {
            "state": state,
            "steps": max_steps,
            "converged": False,
            "energies": energies,
        }


def hamming_distance(a: np.ndarray, b: np.ndarray) -> int:
    """Compute Hamming distance between two binary vectors.

    Args:
        a: First binary vector.
        b: Second binary vector.

    Returns:
        Number of positions where a and b differ.
    """
    return int(np.sum(a != b))


def corrupt_pattern(pattern: np.ndarray, n_flips: int) -> np.ndarray:
    """Corrupt a pattern by flipping n_flips random bits.

    Args:
        pattern: Original binary pattern.
        n_flips: Number of bits to flip.

    Returns:
        Corrupted copy of the pattern.
    """
    corrupted = pattern.copy()
    flip_indices = np.random.choice(len(pattern), size=n_flips, replace=False)
    corrupted[flip_indices] = 1 - corrupted[flip_indices]
    return corrupted


# Quick test
net = HopfieldNetwork(n_neurons=8)
p1 = np.array([1, 1, 1, 1, 0, 0, 0, 0])
p2 = np.array([1, 0, 1, 0, 1, 0, 1, 0])
net.store(np.array([p1, p2]))

print("Weight matrix T_ij:")
print(net.weights)
print(f"\nT_ij is symmetric: {np.allclose(net.weights, net.weights.T)}")
print(f"T_ii = 0: {np.allclose(np.diag(net.weights), 0)}")
print(f"\nEnergy of p1: {net.energy(p1):.1f}")
print(f"Energy of p2: {net.energy(p2):.1f}")

## Part 2: Energy Monotonic Decrease / 에너지 단조 감소 시각화

Hopfield의 핵심 증명: 비동기 갱신 시 $\Delta E \leq 0$ — 에너지는 절대 증가하지 않습니다 (Eq. 8).
이를 실제로 관찰합니다: 무작위 초기 상태에서 시작하여 에너지가 어떻게 변하는지 추적합니다.

Hopfield's key proof: under asynchronous update, $\Delta E \leq 0$ — energy never increases (Eq. 8).
We observe this empirically: starting from random initial states and tracking energy changes.

In [ ]:
# Store 5 random patterns in a 30-neuron network (matching paper's N=30)
N = 30
n_patterns = 5
net = HopfieldNetwork(N)
patterns = np.random.randint(0, 2, size=(n_patterns, N))
net.store(patterns)

# Run retrieval from 5 different random initial states, tracking energy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: energy traces for multiple random starts
for trial in range(5):
    probe = np.random.randint(0, 2, size=N)
    result = net.retrieve(probe, max_steps=500, track_energy=True)
    axes[0].plot(result["energies"], alpha=0.8, label=f"Trial {trial + 1}")

axes[0].set_xlabel("Update step")
axes[0].set_ylabel("Energy E")
axes[0].set_title("Energy monotonically decreases\n(에너지 단조 감소)")
axes[0].legend(fontsize=9)

# Right: verify ΔE ≤ 0 for all steps across many trials
all_deltas = []
for _ in range(50):
    probe = np.random.randint(0, 2, size=N)
    result = net.retrieve(probe, max_steps=300, track_energy=True)
    energies = result["energies"]
    deltas = np.diff(energies)
    all_deltas.extend(deltas)

all_deltas = np.array(all_deltas)
axes[1].hist(all_deltas, bins=50, edgecolor="black", alpha=0.7)
axes[1].axvline(x=0, color="red", linestyle="--", linewidth=2, label="ΔE = 0")
axes[1].set_xlabel("ΔE (energy change per step)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"All ΔE ≤ 0: {np.all(all_deltas <= 1e-10)}\n"
                   f"(max ΔE = {all_deltas.max():.6f})")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Total updates checked: {len(all_deltas)}")
print(f"Any ΔE > 0? {np.any(all_deltas > 1e-10)}")
print(f"Max ΔE: {all_deltas.max():.10f}")

## Part 3: Pattern Storage & Retrieval Demo / 패턴 저장과 복원 데모

5×6 격자(30 뉴런)에 문자 패턴을 저장하고, 손상된 입력에서 원래 패턴을 복원하는 과정을 시각화합니다.

We store letter patterns on a 5×6 grid (30 neurons) and visualize how the network recovers original patterns from corrupted inputs.

In [ ]:
# Define letter patterns on a 5x6 grid (30 neurons)
# Each letter is a 6-row x 5-col binary image, flattened to 30 bits

LETTER_H = np.array([
    1, 0, 0, 0, 1,
    1, 0, 0, 0, 1,
    1, 1, 1, 1, 1,
    1, 0, 0, 0, 1,
    1, 0, 0, 0, 1,
    1, 0, 0, 0, 1,
])

LETTER_A = np.array([
    0, 1, 1, 1, 0,
    1, 0, 0, 0, 1,
    1, 1, 1, 1, 1,
    1, 0, 0, 0, 1,
    1, 0, 0, 0, 1,
    1, 0, 0, 0, 1,
])

LETTER_I = np.array([
    1, 1, 1, 1, 1,
    0, 0, 1, 0, 0,
    0, 0, 1, 0, 0,
    0, 0, 1, 0, 0,
    0, 0, 1, 0, 0,
    1, 1, 1, 1, 1,
])

# Store patterns
net = HopfieldNetwork(30)
letter_patterns = np.array([LETTER_H, LETTER_A, LETTER_I])
net.store(letter_patterns)

# Corrupt each pattern and retrieve
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
cmap = ListedColormap(["white", "black"])
labels = ["H", "A", "I"]

for row, (pattern, label) in enumerate(zip(letter_patterns, labels)):
    # Original
    axes[row, 0].imshow(pattern.reshape(6, 5), cmap=cmap, vmin=0, vmax=1)
    axes[row, 0].set_title(f"Original '{label}'\n(저장된 패턴)")

    # Corrupted (flip 7 bits ~ 23%)
    corrupted = corrupt_pattern(pattern, n_flips=7)
    axes[row, 1].imshow(corrupted.reshape(6, 5), cmap=cmap, vmin=0, vmax=1)
    hd = hamming_distance(pattern, corrupted)
    axes[row, 1].set_title(f"Corrupted\n(Hamming dist = {hd})")

    # Retrieved
    result = net.retrieve(corrupted, max_steps=1000)
    axes[row, 2].imshow(result["state"].reshape(6, 5), cmap=cmap, vmin=0, vmax=1)
    hd_result = hamming_distance(pattern, result["state"])
    axes[row, 2].set_title(f"Retrieved\n(Hamming dist = {hd_result}, "
                           f"steps = {result['steps']})")

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("Hopfield Network: Pattern Retrieval from Corrupted Input\n"
             "(손상된 입력에서 패턴 복원)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Part 4: Attractor Basins / 끌개 영역 시각화

에너지 지형(energy landscape)의 개념을 시각화합니다. 2D 투영을 사용하여, 저장된 패턴들이 에너지 극소(골짜기)에 위치하고, 주변 상태들이 이 골짜기로 수렴하는 것을 보여줍니다.

We visualize the energy landscape concept. Using 2D projection, we show that stored patterns sit at energy minima (valleys) and nearby states converge toward these valleys.

In [ ]:
# Visualize attractor basins: which stored pattern does each random start converge to?
N = 30
n_patterns = 3
net = HopfieldNetwork(N)
patterns = np.random.randint(0, 2, size=(n_patterns, N))
net.store(patterns)

n_trials = 200
initial_states = []
final_patterns = []  # Index of nearest stored pattern after convergence

for _ in range(n_trials):
    probe = np.random.randint(0, 2, size=N)
    initial_states.append(probe.copy())
    result = net.retrieve(probe, max_steps=1000)
    final = result["state"]

    # Find which stored pattern the final state is closest to
    distances = [hamming_distance(final, p) for p in patterns]
    final_patterns.append(np.argmin(distances))

# Project states to 2D using Hamming distance to first two patterns
initial_states = np.array(initial_states)
x = np.array([hamming_distance(s, patterns[0]) for s in initial_states])
y = np.array([hamming_distance(s, patterns[1]) for s in initial_states])
final_patterns = np.array(final_patterns)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ["#e41a1c", "#377eb8", "#4daf4a"]
markers = ["o", "s", "^"]

for i in range(n_patterns):
    mask = final_patterns == i
    ax.scatter(x[mask], y[mask], c=colors[i], marker=markers[i],
               alpha=0.5, s=40, label=f"→ Pattern {i+1}")

# Mark stored patterns
for i, p in enumerate(patterns):
    px = hamming_distance(p, patterns[0])
    py = hamming_distance(p, patterns[1])
    ax.scatter(px, py, c=colors[i], marker="*", s=400, edgecolors="black",
               linewidths=2, zorder=5)

ax.set_xlabel("Hamming distance to Pattern 1\n(패턴 1까지 Hamming 거리)")
ax.set_ylabel("Hamming distance to Pattern 2\n(패턴 2까지 Hamming 거리)")
ax.set_title("Attractor Basins: Where Random States Converge\n"
             "(끌개 영역: 무작위 상태의 수렴 목적지)")
ax.legend(title="Converged to / 수렴 목적지")
plt.tight_layout()
plt.show()

# Count basin sizes
for i in range(n_patterns):
    count = np.sum(final_patterns == i)
    print(f"Pattern {i+1}: {count}/{n_trials} trials "
          f"({100*count/n_trials:.0f}%) converged here")

## Part 5: Storage Capacity — $\sim 0.15N$ Rule / 저장 용량 — $\sim 0.15N$ 규칙

논문의 핵심 결과 중 하나: $N$개 뉴런 네트워크는 약 $0.15N$개의 패턴까지 안정적으로 저장할 수 있습니다.
이를 실험적으로 검증합니다: $n$을 증가시키며 완벽 복원(Hamming distance = 0) 비율을 측정합니다.

One of the paper's key results: a network of $N$ neurons can stably store about $0.15N$ patterns.
We verify experimentally: increasing $n$ and measuring the rate of perfect retrieval (Hamming distance = 0).

In [ ]:
# Reproduce capacity experiment: N=100, vary n from 1 to 25
N = 100
n_values = list(range(1, 26))
n_trials_per = 20
perfect_retrieval_rates = []

for n in n_values:
    perfect_count = 0
    total_tests = 0

    for trial in range(n_trials_per):
        net = HopfieldNetwork(N)
        patterns = np.random.randint(0, 2, size=(n, N))
        net.store(patterns)

        # Test retrieval of each stored pattern (with small corruption)
        for p_idx in range(n):
            corrupted = corrupt_pattern(patterns[p_idx], n_flips=5)
            result = net.retrieve(corrupted, max_steps=2000)
            if hamming_distance(result["state"], patterns[p_idx]) == 0:
                perfect_count += 1
            total_tests += 1

    perfect_retrieval_rates.append(perfect_count / total_tests)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(n_values, perfect_retrieval_rates, "bo-", linewidth=2, markersize=8)
ax.axvline(x=0.15 * N, color="red", linestyle="--", linewidth=2,
           label=f"0.15N = {0.15*N:.0f} (theoretical limit)")
ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.5)

ax.set_xlabel("Number of stored patterns (n)\n(저장된 패턴 수)")
ax.set_ylabel("Perfect retrieval rate\n(완벽 복원 비율)")
ax.set_title(f"Storage Capacity Experiment (N = {N})\n"
             f"(저장 용량 실험)")
ax.legend(fontsize=12)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print(f"\nTheoretical capacity: 0.15 × {N} = {0.15*N:.0f} patterns")
# Find empirical threshold (where rate drops below 50%)
for i, rate in enumerate(perfect_retrieval_rates):
    if rate < 0.5:
        print(f"Empirical threshold (rate < 50%): n = {n_values[i]}")
        break

## Part 6: Hamming Distance Analysis / Hamming 거리 분석

논문 p.2557의 핵심 실험 재현: 초기 상태의 Hamming distance에 따른 올바른 기억 복원 확률을 측정합니다.

Reproducing the key experiment from p.2557: measuring the probability of correct memory retrieval as a function of initial state Hamming distance.

In [ ]:
# Hamming distance vs retrieval success (paper: N=30, n=5)
N = 30
n = 5
n_trials = 100

net = HopfieldNetwork(N)
patterns = np.random.randint(0, 2, size=(n, N))
net.store(patterns)

flip_range = range(0, N + 1)
success_rates = []

for n_flips in flip_range:
    successes = 0
    for _ in range(n_trials):
        # Pick a random stored pattern and corrupt it
        p_idx = np.random.randint(n)
        if n_flips == 0:
            corrupted = patterns[p_idx].copy()
        else:
            corrupted = corrupt_pattern(patterns[p_idx], n_flips=min(n_flips, N))

        result = net.retrieve(corrupted, max_steps=1000)

        # Success = converged to the correct pattern
        if hamming_distance(result["state"], patterns[p_idx]) == 0:
            successes += 1

    success_rates.append(successes / n_trials)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(flip_range), success_rates, "bo-", markersize=5, linewidth=2)
ax.axhline(y=1/n, color="red", linestyle="--", alpha=0.7,
           label=f"Random chance = 1/{n} = {1/n:.2f}")
ax.axvline(x=N//6, color="green", linestyle=":", alpha=0.7,
           label=f"~N/6 = {N//6} (paper's approximate boundary)")

ax.set_xlabel("Number of flipped bits (= Hamming distance from stored pattern)\n"
              "(뒤집힌 비트 수 = 저장 패턴으로부터 Hamming 거리)")
ax.set_ylabel("Correct retrieval probability\n(올바른 복원 확률)")
ax.set_title(f"Retrieval Success vs Hamming Distance (N={N}, n={n})\n"
             f"(복원 성공률 vs Hamming 거리)")
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print(f"At 0 flips: {success_rates[0]:.2f}")
print(f"At 5 flips: {success_rates[5]:.2f}")
print(f"At 10 flips: {success_rates[10]:.2f}")
print(f"At 15 flips (N/2): {success_rates[15]:.2f}")

## Part 7: Spurious States / 가짜 기억 (Spurious States)

Hopfield 네트워크에는 저장된 패턴 외에도 **가짜 기억(spurious states)** — 의도하지 않은 에너지 극소점 — 이 존재합니다.
가장 대표적인 것은 저장된 패턴들의 **혼합 상태(mixture states)** 입니다.

Besides stored patterns, Hopfield networks have **spurious states** — unintended energy minima.
The most common are **mixture states** of stored patterns.

In [ ]:
# Detect spurious states by running many random starts and checking
# if the final state matches any stored pattern
N = 30
n = 5
net = HopfieldNetwork(N)
patterns = np.random.randint(0, 2, size=(n, N))
net.store(patterns)

n_trials = 500
stored_count = 0
spurious_count = 0
spurious_states = []
stored_energies = [net.energy(p) for p in patterns]

for _ in range(n_trials):
    probe = np.random.randint(0, 2, size=N)
    result = net.retrieve(probe, max_steps=2000)
    final = result["state"]

    # Check if final state matches any stored pattern
    is_stored = any(hamming_distance(final, p) == 0 for p in patterns)
    # Also check complements (inverted patterns are also stable)
    is_complement = any(hamming_distance(final, 1 - p) == 0 for p in patterns)

    if is_stored:
        stored_count += 1
    elif is_complement:
        stored_count += 1  # Complements are expected attractors
    else:
        spurious_count += 1
        spurious_states.append(final.copy())

print(f"Results from {n_trials} random starts:")
print(f"  Converged to stored pattern (or complement): {stored_count} "
      f"({100*stored_count/n_trials:.1f}%)")
print(f"  Converged to SPURIOUS state: {spurious_count} "
      f"({100*spurious_count/n_trials:.1f}%)")

if spurious_states:
    # Analyze spurious states
    print(f"\nAnalysis of {min(5, len(spurious_states))} spurious states:")
    for idx, ss in enumerate(spurious_states[:5]):
        e = net.energy(ss)
        dists = [hamming_distance(ss, p) for p in patterns]
        print(f"  Spurious #{idx+1}: energy={e:.1f}, "
              f"Hamming dists to stored={dists}")

    print(f"\nStored pattern energies: "
          f"{[f'{e:.1f}' for e in stored_energies]}")

    # Visualize: energy comparison
    fig, ax = plt.subplots(figsize=(10, 5))
    sp_energies = [net.energy(s) for s in spurious_states]

    ax.hist(stored_energies, bins=10, alpha=0.7, color="blue",
            label=f"Stored patterns (n={n})", edgecolor="black")
    ax.hist(sp_energies, bins=20, alpha=0.7, color="red",
            label=f"Spurious states (n={len(spurious_states)})", edgecolor="black")
    ax.set_xlabel("Energy")
    ax.set_ylabel("Count")
    ax.set_title("Energy: Stored Patterns vs Spurious States\n"
                 "(저장된 패턴 vs 가짜 기억의 에너지)")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("\nNo spurious states found (all converged to stored patterns).")

## Part 8: Connection to Modern Methods / 현대 기법과의 연결

Hopfield 네트워크는 현대 딥러닝의 여러 핵심 개념의 직접적 선조입니다:

The Hopfield network is a direct ancestor of several core concepts in modern deep learning:

| Hopfield (1982) | 현대 기법 / Modern Method | 연결 / Connection |
|---|---|---|
| Energy function $E$ | Loss function in deep learning | 네트워크 학습 = 에너지/손실 최소화 / Network training = energy/loss minimization |
| Attractor dynamics | Denoising in diffusion models (DDPM, 2020) | 손상된 입력 → 깨끗한 출력으로의 반복적 정제 / Iterative refinement from corrupted → clean |
| Hebbian $T_{ij}$ | Attention weights in Transformers | 패턴 간 유사도 기반 연결 강도 / Similarity-based connection strength |
| Boltzmann Machine (1985) | RBMs → Deep Belief Nets (2006) | 확률론적 Hopfield → 딥러닝 부흥 / Stochastic Hopfield → deep learning revival |
| Modern Hopfield (2020) | Transformer attention | Ramsauer et al. showed exponential capacity Hopfield = attention |

아래에서 Hopfield 네트워크와 현대 **연속 Hopfield 네트워크(Modern Hopfield Network)** 를 비교합니다.

Below we compare the classic Hopfield network with the **Modern Hopfield Network** (Ramsauer et al., 2020).

In [ ]:
# Modern Hopfield Network: continuous states with exponential capacity
# Ramsauer et al. (2020) showed that the update rule:
#   x_new = softmax(beta * X^T @ x) @ X
# is equivalent to Transformer attention and has exponential storage capacity.

def modern_hopfield_retrieve(
    patterns: np.ndarray,
    query: np.ndarray,
    beta: float = 1.0,
    n_steps: int = 5,
) -> list[np.ndarray]:
    """Modern (continuous) Hopfield network retrieval.

    Uses the softmax-based update rule from Ramsauer et al. (2020).

    Args:
        patterns: Stored patterns, shape (n_patterns, dim).
        query: Query vector, shape (dim,).
        beta: Inverse temperature (sharpness of softmax).
        n_steps: Number of update iterations.

    Returns:
        List of states at each step.
    """
    x = query.copy().astype(float)
    trajectory = [x.copy()]

    for _ in range(n_steps):
        # Compute attention scores: softmax(beta * X^T @ x)
        scores = beta * patterns @ x  # shape: (n_patterns,)
        scores -= scores.max()  # Numerical stability
        weights = np.exp(scores) / np.exp(scores).sum()
        # Update: weighted combination of stored patterns
        x = weights @ patterns
        trajectory.append(x.copy())

    return trajectory


# Compare classic vs modern Hopfield
N = 50
n = 3
np.random.seed(123)

# Generate patterns (continuous for modern, binary for classic)
patterns_bin = np.random.randint(0, 2, size=(n, N))
patterns_cont = (2 * patterns_bin - 1).astype(float)  # {-1, +1}

# Create a corrupted query (30% noise)
target_idx = 0
query_bin = corrupt_pattern(patterns_bin[target_idx], n_flips=15)
query_cont = (2 * query_bin - 1).astype(float)

# Classic Hopfield
net = HopfieldNetwork(N)
net.store(patterns_bin)
result_classic = net.retrieve(query_bin, max_steps=500, track_energy=True)

# Modern Hopfield (various beta)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Top-left: Classic energy trace
axes[0, 0].plot(result_classic["energies"], "b-", linewidth=2)
axes[0, 0].set_xlabel("Step")
axes[0, 0].set_ylabel("Energy")
axes[0, 0].set_title("Classic Hopfield: Energy trace\n(고전 Hopfield: 에너지 추적)")

# Top-right: Modern Hopfield convergence for different beta
for beta in [0.1, 0.5, 1.0, 5.0]:
    trajectory = modern_hopfield_retrieve(patterns_cont, query_cont,
                                           beta=beta, n_steps=10)
    # Measure similarity to target at each step
    sims = [np.dot(s, patterns_cont[target_idx]) / (np.linalg.norm(s) *
            np.linalg.norm(patterns_cont[target_idx]) + 1e-10)
            for s in trajectory]
    axes[0, 1].plot(sims, "o-", label=f"β={beta}")

axes[0, 1].set_xlabel("Step")
axes[0, 1].set_ylabel("Cosine similarity to target\n(대상 패턴과의 코사인 유사도)")
axes[0, 1].set_title("Modern Hopfield: Convergence\n(현대 Hopfield: 수렴)")
axes[0, 1].legend()
axes[0, 1].set_ylim(0, 1.05)

# Bottom-left: Classic capacity (already computed above, use summary)
classic_caps = []
for n_pat in [3, 5, 8, 10, 15, 20]:
    successes = 0
    for _ in range(50):
        net_tmp = HopfieldNetwork(N)
        pats = np.random.randint(0, 2, size=(n_pat, N))
        net_tmp.store(pats)
        p_idx = np.random.randint(n_pat)
        corrupted = corrupt_pattern(pats[p_idx], n_flips=5)
        res = net_tmp.retrieve(corrupted, max_steps=1000)
        if hamming_distance(res["state"], pats[p_idx]) == 0:
            successes += 1
    classic_caps.append((n_pat, successes / 50))

n_pats, rates = zip(*classic_caps)
axes[1, 0].bar(range(len(n_pats)), rates, tick_label=[str(x) for x in n_pats],
               color="steelblue", edgecolor="black")
axes[1, 0].axhline(y=0.5, color="red", linestyle="--")
axes[1, 0].set_xlabel("Number of patterns (n)")
axes[1, 0].set_ylabel("Retrieval rate")
axes[1, 0].set_title(f"Classic: Capacity ~0.15N={0.15*N:.0f}\n"
                      f"(고전: 용량 ~0.15N)")

# Bottom-right: Summary comparison table
axes[1, 1].axis("off")
table_data = [
    ["Feature", "Classic (1982)", "Modern (2020)"],
    ["States", "Binary {0,1}", "Continuous ℝ"],
    ["Update", "Async threshold", "Softmax attention"],
    ["Capacity", "~0.15N", "~exp(N)"],
    ["Convergence", "Energy ↓", "Energy ↓ (proved)"],
    ["Link to DL", "Boltzmann Machine", "Transformer attention"],
]
table = axes[1, 1].table(cellText=table_data, loc="center",
                          cellLoc="center", colWidths=[0.3, 0.35, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
# Header row styling
for j in range(3):
    table[0, j].set_facecolor("#4472C4")
    table[0, j].set_text_props(color="white", fontweight="bold")
axes[1, 1].set_title("Classic vs Modern Hopfield\n(고전 vs 현대 Hopfield)", pad=20)

plt.tight_layout()
plt.show()

print("Key insight: Modern Hopfield's softmax update is mathematically")
print("equivalent to the attention mechanism in Transformers (Vaswani et al., 2017).")
print("This connects the 1982 paper directly to the foundation of modern LLMs.")